# 00 — Check data access

Confirms the OneDrive sensor folder is readable and that daily files have the expected column structure and approximately 2-minute timestamp spacing. This is a pre-flight sanity check; it produces no saved output.

In [1]:
import sys
import re
from pathlib import Path
import matplotlib.pyplot as plt


sys.path.append(str(Path.cwd().parent))
from src.config import *
import pandas as pd

print("Folder exists:", DATA_DIR.exists())

# One spelling of the units and the species, used in every figure.
UNITS = r"$\mu$g m$^{-3}$"
PM25  = r"PM$_{2.5}$"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False,
})

Folder exists: True


### Top-level contents

The raw folder holds one directory per sensor. The 67 `SL`-prefixed directories include all units on disk — indoor, test, and undeployed sensors are present but excluded from the study set in NB01, which retains 47 outdoor Leeds sensors.

In [2]:
print("Top-level contents:")
for item in sorted(DATA_DIR.iterdir()):
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"  [{kind}] {item.name}")

sl_dirs = [d for d in DATA_DIR.iterdir() if d.is_dir() and re.match(r"SL\d", d.name)]
print(f"\nSL directories on disk: {len(sl_dirs)}  (includes indoor and test units; 47 become the study set in NB01)")

Top-level contents:
  [FILE] desktop.ini
  [DIR ] location
  [DIR ] plots
  [FILE] purpleair_data_format.txt
  [DIR ] R_code
  [DIR ] SEE_2
  [FILE] sensor_summary(250910).csv
  [FILE] sensor_summary(260108).csv
  [FILE] sensor_summary(260310).csv
  [FILE] sensor_summary.csv
  [DIR ] SL001_Sunnyview_Terrace
  [DIR ] SL002
  [DIR ] SL003_Corn_Exchange_Cabinet
  [DIR ] SL004
  [DIR ] SL005_Kirkstall_Valley_Primary
  [DIR ] SL006
  [DIR ] SL007_Ninelands_Primary
  [DIR ] SL008_Westgate_Primary
  [DIR ] SL009
  [DIR ] SL010_Wetherby_Racecourse
  [DIR ] SL011
  [DIR ] SL012_Morley
  [DIR ] SL013_Knowsthorpe_Gate
  [DIR ] SL014_Pudsey
  [DIR ] SL015_-_Garforth_Main_Street
  [DIR ] SL016_Scholes_Main_Street
  [DIR ] SL018_East_Ardsley_A650
  [DIR ] SL019_Cornwall_Crescent_Rothwell
  [DIR ] SL020
  [DIR ] SL021_Kippax_Church_Lane
  [DIR ] SL022_Church_Lane_Horsforth
  [DIR ] SL023_Nelson_Street_Otley
  [DIR ] SL024_Wetherby_Town_Centre
  [DIR ] SL025_-_Oakwood
  [DIR ] SL026
  [DIR ] SL027_-_S

### First populated daily file

Daily files are named `YYYY-MM-DD.csv`. The check finds the first non-empty one and verifies the expected columns and median timestamp spacing.

In [3]:
first_populated, empty_count, checked = None, 0, 0
for p in DATA_DIR.rglob("*.csv"):
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}\.csv", p.name):
        checked += 1
        df = pd.read_csv(p)
        if len(df) > 0:
            first_populated = (p, df)
            break
        empty_count += 1
        if checked > 400:
            break

print(f"Empty daily files skipped: {empty_count}\n")

if first_populated is None:
    print("No populated daily file found in first batch checked.")
else:
    p, df = first_populated
    print(f"File: {p}")
    print(f"Rows: {len(df)}  (720 = full day at 2-min spacing)\n")

    cols = [DATE_COL, PM_A, PM_B, TEMP_COL, RH_COL]
    print(df[cols].head(5).to_string())

    ts = pd.to_datetime(df[DATE_COL], utc=True)
    median_gap = ts.diff().dropna().median()
    print(f"\nMedian timestamp spacing: {median_gap}  (expected ~0 days 00:02:00)")

Empty daily files skipped: 0

File: C:\Users\user\OneDrive - University of Leeds\Dissertation Data\SEE AQ Projects-PURPLEAIR - sensor_data\Wetherby_Town_WB02\2026\01\2026-01-01.csv
Rows: 643  (720 = full day at 2-min spacing)

                   date  PM2.5 A (CF=ATM) (ug/m3)  PM2.5 B (CF=ATM) (ug/m3)  Temperature (F)  Humidity (%)
0  2026-01-01T00:00:39Z                       2.4                       2.8              NaN           NaN
1  2026-01-01T00:02:39Z                       2.0                       2.4              NaN           NaN
2  2026-01-01T00:04:39Z                       2.1                       1.8              NaN           NaN
3  2026-01-01T00:06:39Z                       2.0                       2.0              NaN           NaN
4  2026-01-01T00:08:39Z                       2.2                       1.8              NaN           NaN

Median timestamp spacing: 0 days 00:02:00  (expected ~0 days 00:02:00)
